# Hansen Ch.8 Restricted Estimation — 计算

**Chapter 8 Restricted Estimation**

完整推导与**面向初学者的详细注释**见同目录 `Hansen_Ch08_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：**Exercise 8.19**（CLS 与有效 MD 实操）。

> **写给只学过李子奈/陈强的同学：** 本章讲**当经济理论给出约束 $R'\beta=c$ 时如何利用它**。两套等价工具：
> - **CLS（约束最小二乘）**：约束下最小化 SSE，Stata `cnsreg`。
> - **MD（最小距离）**：把无约束 $\hat\beta$ 投影到约束集合，权重 $W$。CLS 是 MD 的特例（$W=\hat Q_{XX}$）。
>
> 核心直觉——**施加正确约束让方差下降**（"减去一个半正定项"）：
> $$V_{\tilde\beta,\text{有效}}=V_\beta-V_\beta R(R'V_\beta R)^{-1}R'V_\beta\le V_\beta.$$
> 但约束**为真**才有此好处；约束**错**了会引入偏差（且让 SSE 上升，见 Ex 8.21）。
> 有效 MD 用 $W=V_\beta^{-1}$（方差倒数作权重），同方差下退化为 CLS，异方差下优于 CLS。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

def ols_hc3(y, X):
    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    XXinv = np.linalg.inv(X.T @ X)
    h = np.sum(X * (X @ XXinv), axis=1)
    u = X * (e / np.clip(1 - h, 1e-12, None))[:, None]
    V = XXinv @ (u.T @ u) @ XXinv
    return beta, e, V, n, k

CPS = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/cps09mar/cps09mar.xlsx")
df = pd.read_excel(CPS)
df["experience"] = df["age"] - df["education"] - 6
df["lwage"] = np.log(df["earnings"] / (df["hours"] * df["week"]))
df["exp2"] = (df["experience"] ** 2) / 100
s = df[(df.race == 1) & (df.female == 0) & (df.hisp == 1)].copy().reset_index(drop=True)
for code, name in [(1, "m1"), (2, "m2"), (3, "m3"), (4, "wid"), (5, "div"), (6, "sep")]:
    s[name] = (s.marital == code).astype(float)
y = s.lwage.to_numpy()
X = np.column_stack([
    s.education, s.experience, s.exp2, s.m1, s.m2, s.m3, s.wid, s.div, s.sep, np.ones(len(s))
])
names = ["edu", "exp", "exp2", "m1", "m2", "m3", "wid", "div", "sep", "int"]
beta, e, V, n, k = ols_hc3(y, X)
print("n =", n)
print(pd.DataFrame({"beta": beta, "HC3_SE": np.sqrt(np.diag(V))}, index=names))

# CLS: m1=wid, div=sep
Xcls = np.column_stack([
    s.education, s.experience, s.exp2, s.m1 + s.wid, s.m2, s.m3, s.div + s.sep, np.ones(len(s))
])
bcls, _, Vcls, _, _ = ols_hc3(y, Xcls)
print("\nCLS (reparameterized):", bcls)

# EMD
R = np.array([[0, 0, 0, 1, 0, 0, -1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, -1, 0]], float)
bmd = beta - V @ R.T @ np.linalg.inv(R @ V @ R.T) @ (R @ beta)
print("\nEMD:")
print(pd.Series(bmd, index=names))
print("d(exp)/d at 0 and 50:", beta[1], beta[1] + beta[2])
